# Quantum Pipeline

Define small feature map (angle encoding, ZZFeatureMap).

Define variational ansatz (1–3 layers).

Measure expectation values → feature vector.

Train classical classifier on these features.

In [58]:
import pennylane as qml
import pennylane.numpy as np
import pennylane_lightning
from pennylane.optimize import NesterovMomentumOptimizer

import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, auc
# from joblib import Parallel, delayed
import joblib
from multiprocessing import Pool
import itertools
import time
import pandas as pd
# import numpy as np
import os
from dotenv import load_dotenv

from qiskit_ibm_runtime import QiskitRuntimeService
from qfm import zz_feature_map, var_ansatz  # shared with quantum_feature_maps_tests.ipynb

In [59]:
data = np.load("data/mnist01_pca4.npz")    #from dataprep notebook

X_train, y_train = data["X_train"], data["y_train"]  
X_test, y_test = data["X_test"], data["y_test"]

#scaling x to fit within [0, π] so the feature map can use meaninful angles instead of angles between [-10,10]
X_max = np.max(np.abs(X_train))
X_train = (np.pi * X_train) / X_max
X_test  = (np.pi * X_test) / X_max


# x = features/ dimensions --> 4
# y = labels
n_qubits=4

#backened 
# dev= qml.device("default.qubit", wires=n_qubits)
dev = qml.device("lightning.qubit", wires=n_qubits)    #faster simulator, uses compiled c++ 


print("PCA reduced shape xtrain:  ", X_train.shape)
print("PCA reduced shape xtest:  ", X_test.shape)
print("ytrain labels: ", y_train.shape)
print("ytest labels: ",y_test.shape)


PCA reduced shape xtrain:   (12665, 4)
PCA reduced shape xtest:   (2115, 4)
ytrain labels:  (12665,)
ytest labels:  (2115,)


## Quantum Feature Map

Classical data is transformed into quantum states. For the main pipeline, I will be using PennyLane as it is easier to build custom circuits and I can compare feature maps in a single framework. Pennylane also supports Qiskit backeneds for a future plugin to run on IBM devices. 


<br>

This is a quantum map $\phi(\mathbf{x})$ from classical feature vector $\mathbf{x}$ to quantum state $|\Phi(\mathbf{x})\rangle\langle\Phi(\mathbf{x})|$ in the Hilbert space, achieved through applying the parameterized unitary operator $\mathcal{U}_{\Phi(\mathbf{x})}$ to the initial state $|0\rangle^{n}$, n being the num of qubits for encoding. This is done so quantum algorithms can use the data. 

Unitary operations are a building block of Qcircuits, where a matrix U is unitary if $$U^\dagger U = U U^\dagger = I$$ 

Some examples are the Pauli X Gate or Hadamard Gate, or in this case, the Rotation Gate $R_y(\theta)$. 
$$
R_y(\theta) = \begin{bmatrix}\cos(\theta/2) & -\sin(\theta/2) \\ \sin(\theta/2) & \cos(\theta/2)\end{bmatrix}, \quad
R_y^\dagger(\theta) = R_y(-\theta), \quad
R_y(\theta) R_y^\dagger(\theta) = I
$$


### ZZ Map Manual - PennyLane

For a ZZ Feature Map, PennyLane does not already have a built in function so we have to do it manually using Hadamard gates, along wiht RZ and ZZ rotations for each qubit pair. This just makes it simpler to integrate over the course of this project. The Qiskit version of this gate is shown in my quantum testing ground. 

Also, out of all encoding methods I could've picked, Angle Encoding seemed to be the most simple and effective when working with circuits (perfect when I am working on a simple project).

The ZZ feature map is a version of the Pauli feature map, which includes entangling gates (ZZ interactions thorugh controlled-phase gates), done after initial data encoding rotations. This is done to allow feature map to capture any potential correlations between features and then maps the data to a more complex hilbert space. 

There are layers of Hadamard gates, single qubit phase rotations, and 2 qubit controlled phase rotations. 

<br>

Hadamard Gates

Creates equal superposition of either state within a qubit

$$
H = \tfrac{1}{\sqrt{2}}
\begin{pmatrix}
1 & 1 \\[6pt]
1 & -1
\end{pmatrix}
$$

RZ Gates 

Single qubit phase rotation around Z axis, diagonal gate

$$
R_Z(\phi) 
= \exp\!\left(-i \tfrac{\phi}{2} Z\right) 
= 
\begin{pmatrix}
e^{-i\phi/2} & 0 \\[6pt]
0 & e^{\,i\phi/2}
\end{pmatrix}
$$


CNOT Gates

Flips target qubit if control qubit is 1.  --> encodes pairwise correlations

$$
\text{CNOT} = CX =
\begin{pmatrix}
1 & 0 & 0 & 0 \\[6pt]
0 & 1 & 0 & 0 \\[6pt]
0 & 0 & 0 & 1 \\[6pt]
0 & 0 & 1 & 0
\end{pmatrix}
$$

<br>

ZZ Feature Map

Encodes the pairwise correlations between features (by constructing a unitary transformation)

$$
U_{\mathrm{ZZ}}(x) = \exp\left( i \sum_i x_i Z_i + i \sum_{i<j} x_i x_j Z_i Z_j \right)
$$


In [60]:
sample_vec = X_train[0]      #1st of 12665
print("Example feature vector: ", sample_vec)  


Example feature vector:  [-0.7779276  -0.57059664 -0.12868546 -0.11485348]


In [61]:
# zz_feature_map now lives in src/qfm/feature_maps.py (imported above)

@qml.qnode(dev)
def qnode_measure(x):
    zz_feature_map(x, wires=range(4), reps=1)
    return qml.state()


print(f"ZZFeatureMap state vector: \n {qnode_measure(sample_vec)} \n") 
print(qml.draw(qnode_measure, level='device')(sample_vec))    


ZZFeatureMap state vector: 
 [ 0.12222394+0.21808555j  0.16314236+0.18943223j  0.1394597 +0.20748733j
  0.18799265+0.16479916j  0.14472847+0.20384717j  0.18236686+0.17100389j
  0.2092889 +0.1367412j   0.2373507 +0.07851525j  0.23104784+0.09548243j
  0.24541871+0.04764094j  0.23801256+0.07648543j  0.24966647+0.01290938j
  0.02012213-0.24918888j -0.02982166-0.24821496j -0.07246733-0.23926656j
 -0.13138659-0.21269124j] 

0: ──H──RZ(-1.56)─╭●───────────╭●─────────────────────────────────┤ ╭State
1: ──H──RZ(-1.14)─╰X──RZ(0.89)─╰X─╭●───────────╭●─────────────────┤ ├State
2: ──H──RZ(-0.26)─────────────────╰X──RZ(0.15)─╰X─╭●───────────╭●─┤ ├State
3: ──H──RZ(-0.23)─────────────────────────────────╰X──RZ(0.03)─╰X─┤ ╰State


In this circuit, you can see how each qubit has a Hadamard gate applied (without this, qubits starting in |0⟩ and RZ would only add a global phase - unobservable, as RZ works with relative phase differences between |0⟩ and |1⟩). 

Then, a phase rotation of (x) amount around the Z axis to encode the classical features x[i] as a Z axis phase shift of angle 2x[i].  

Finally, we have the entanglement layers that work through each adjacent pair of qubits to perform phase shifts. Applies controlled phase depending on both qubits and encodes pairwaise correlations into the state. 

**Creates a nonlienar feature map in Hilbert space.**

A CNOT sandwich, which is seen within the multiqubit entanglement layer above, allows for conditional application of a targets rotation, depending on the control qubit. So, in the end, we are maintaining the computational values of the qubit but keeping the phase entanglement of the qubits (it would look the same on the 0,1 level but now hidden quantum phase correlations encode the data)

## Variational Ansatz 

Ansatz: basic architecture of circuit, set of gates that act on specific subsystems with a few assumptions about the appropriate training circuit. Can be generic/problem neutral (hardware ansatz) or can be problem specific. 

Quantum Ansatz: Parameterized Quantum Circuit (PQC) that represents a family of possible quantum stats that are controlled by tunable paramters (normally angles)

$$
|\psi(\theta)\rangle = U(\theta) |0\rangle
$$

ansatz circuit: $$U(\theta)$$ parameters: $$\theta = [\theta_1, \theta_2, \ldots]$$ 


Method used to approimate ground state (lowest energy eingenstate) of quantum system by selecting a trial wavefunction with adjustable parameters. The expectation value of the energy will always be >= to the true ground state energy, so if we optimize the trial wavefunction, we can minimize the expectation value. Built from rotation gates (paramterized by angles) and entangling gates --> these form a layer, we can form many and stakc them to increase expresssiveness. 

**Analogy to CC**

Similar to how a NN layer has weights = rotational angles in the quantum gates. 
- training adjusts the angles to minimize some error/loss 
- patterns to how the gates connect = circuits architecture 

Also similar to deeper NN layers, we can stack them to increase expressiveness


In [62]:
# var_ansatz now lives in src/qfm/ansatz.py (imported above)

#intial params tensors - random, will be trained in ansatz 
params = np.random.randn(2, n_qubits, 3)     # 2 layers, n_qubits =4, 3 euler angles for each rotation axis 

@qml.qnode(dev)
def ansatz_preview(params):
    var_ansatz(params)
    return qml.state()

print(qml.draw(ansatz_preview)(params))


0: ──Rot(1.22,0.66,-0.78)───╭●──Rot(0.18,-0.94,-1.29)─────────────────────── ···
1: ──Rot(2.14,1.47,-0.45)───╰X─╭●──────────────────────Rot(0.24,-0.51,-0.10) ···
2: ──Rot(0.40,-0.05,-0.24)─────╰X─────────────────────╭●──────────────────── ···
3: ──Rot(-0.11,-0.04,-2.18)───────────────────────────╰X──────────────────── ···

0: ··· ─╭●───────────────────────────┤  State
1: ··· ─╰X─────────────────────╭●────┤  State
2: ··· ──Rot(-0.41,0.66,-1.24)─╰X─╭●─┤  State
3: ··· ──Rot(0.45,0.37,-0.52)─────╰X─┤  State


This shows rotation gates on each qubit [0..3], with 3 values inside the parenthesis to represent the 3 euler angles used for the rotation (qml.Rot(θ, φ, λ)). The entanglers (CNOT gates) are the vertical lines and dot connecting the qubits. For instance, the dot on qubit 0 indicates that it is the control and the target is qubit 1. There are 6 CNOT gates (3 per layer), with rotations done on each qubit before applying CNOT gates. 

## Measure Expectation Value

In [63]:
@qml.qnode(dev, diff_method="parameter-shift")      #alllows for batches 
def qnode_measure(x, params):
    # x_batch = np.atleast_2d(x_batch)   #ensures shape [batch,featues]       
    zz_feature_map(x, wires=range(4), reps=1)              #encodes data
    var_ansatz(params)       #transforms encoded state w var ansatz
    #returns expectation values  - explicitly 
    return (
        qml.expval(qml.PauliZ(0)),
        qml.expval(qml.PauliZ(1)),
        qml.expval(qml.PauliZ(2)),
        qml.expval(qml.PauliZ(3)),
    )




def get_quantum_features(X, params):
    feats = []
    for x in X:
        feats.append(qnode_measure(x,params))
    # np.array([qnode_measure(x,params) for x in X])
    return feats           #shape [N,n_qubits]

In [64]:
print(qml.draw(qnode_measure)(sample_vec, params))      #now displays zzfeature map  and var ansatz

0: ──H──RZ(-1.56)─╭●───────────╭●──Rot(1.22,0.66,-0.78)─────────────────────────────────── ···
1: ──H──RZ(-1.14)─╰X──RZ(0.89)─╰X─╭●──────────────────────────────╭●──Rot(2.14,1.47,-0.45) ···
2: ──H──RZ(-0.26)─────────────────╰X─────────────────────RZ(0.15)─╰X─╭●─────────────────── ···
3: ──H──RZ(-0.23)────────────────────────────────────────────────────╰X─────────────────── ···

0: ··· ─╭●─────────Rot(0.18,-0.94,-1.29)────────────────────────────────────────────────── ···
1: ··· ─╰X───────────────────────────────────────────────────────╭●──Rot(0.24,-0.51,-0.10) ···
2: ··· ───────────╭●──────────────────────Rot(0.40,-0.05,-0.24)──╰X─╭●──────────────────── ···
3: ··· ──RZ(0.03)─╰X──────────────────────Rot(-0.11,-0.04,-2.18)────╰X──────────────────── ···

0: ··· ─╭●───────────────────────────┤  <Z>
1: ··· ─╰X─────────────────────╭●────┤  <Z>
2: ··· ──Rot(-0.41,0.66,-1.24)─╰X─╭●─┤  <Z>
3: ··· ──Rot(0.45,0.37,-0.52)─────╰X─┤  <Z>


Here you can see the original ZZ featuremap applied to the qubits: each one has a Hadamard gate for placing them into superpositions, then an RZ gate to rotate the qubits around the Z axis by an angle proportionate to its xi value (encodes the data into it's phase), then uses CNOT entanglers and RZ gates to connect qubits. 

The second layer is the Variational Ansatz, where the rotation gates (with the 3 angles) allow for any single-qubit rotation in the bloch sphere, which makes it possible for the ansatz to explore all possible states. 

The final measurement for each qubit is <Z>, which is the measured expectation value of PauliZ operator on each qubit. They then become a classical feature vector we input into the classic models. 

In [65]:
outputs = np.array([qnode_measure(x, params) for x in X_train[:10]])
print(outputs)

[[ 0.39727361  0.40787416  0.20725898  0.12237862]
 [ 0.31162005 -0.08058577 -0.26121673 -0.08386894]
 [ 0.00167646 -0.0094298   0.07521082 -0.09357185]
 [ 0.00946866 -0.08571466  0.02044567 -0.10491752]
 [-0.01849059 -0.1089239   0.00294434 -0.09595662]
 [ 0.46550855  0.44173919  0.16984248  0.03985003]
 [ 0.29479056 -0.07827767 -0.26271936 -0.08521349]
 [ 0.30370992 -0.45299795 -0.1571399  -0.14193392]
 [-0.02559947  0.21826525 -0.01162351 -0.10379988]
 [ 0.52855019  0.41432453  0.06298149  0.09807753]]


### Training Params



This trains the parameters using the variational ansatz and uses a gradient descent optimizer to help train/optimize the params over 20 epochs (using a MSE loss function). 

This block of code runs for several hours, so I have it save the results to the folder /notebooks/params. The bottom of the code (training loop) will be commented out so it will not run each time. 

We use Mean Squared Error (MSE) cost function, which allows us to see if the parameters are a better (closer) fit than randomly assigned parameters. 

In [66]:
opt = qml.GradientDescentOptimizer(stepsize=0.1)    #tweaks params to minimize cost function
subset = 500  
depth = 2

for i in range(2,5):   # 2-4 qubit circuits 
    print(f"\nTraining {i} qubit circuit")

    dev = qml.device("default.qubit", wires=i, shots=None)
    @qml.qnode(dev)
    def qnode_measure(x, params):
        zz_feature_map(x, wires=range(i), reps=depth)
        var_ansatz(params)  # your variational ansatz should match i qubits
        return [qml.expval(qml.PauliZ(j)) for j in range(i)]

    params = np.random.randn(depth, i, 3)

    # defining a cost function - MSE 
    def cost(params, X, y):         #params (layers, nqubits, 3 euler angles), x=classical data [N, nfeatures], y=labels (0 or 1)
        preds = np.array([qnode_measure(x, params) for x in X])   #for each x, runs zzmap, var ansatz, returns expect value of Z for qubit - [N, nfeatures]
        #loss function
        return np.mean((preds[:,0] - y)**2)  # MSE betwen predicted quantum expect values and classic labels y





    #training loop
    for epoch in range(20):        #optimizes over 50 iterations 
        idx = np.random.permutation(len(X_train))     #ensures we train on different subsets each time
        X_shuffled, y_shuffled = X_train[idx], y_train[idx]

        params = opt.step(lambda p: cost(p, X_shuffled[:subset], y_shuffled[:subset]), params)   #we try to reduce cost while updating params each step 
        cost_val = cost(params, X_train[:subset], y_train[:subset])

        print(f"Epoch: {epoch}, Cost: {cost_val}")


    np.save(f"params/trained_params_{i}.npy", params)    #saves trained numpy array of params to params folder
    print(qml.draw(ansatz_preview)(params))   #recheck 



Training 2 qubit circuit
Epoch: 0, Cost: 1.5750709730420067
Epoch: 1, Cost: 1.4668407051436756
Epoch: 2, Cost: 1.3485858676355664
Epoch: 3, Cost: 1.216888409409759
Epoch: 4, Cost: 1.0969260557160287
Epoch: 5, Cost: 0.9788973074868469
Epoch: 6, Cost: 0.8796901259313504
Epoch: 7, Cost: 0.7796243884853167
Epoch: 8, Cost: 0.6941131732270053
Epoch: 9, Cost: 0.6232843697698444
Epoch: 10, Cost: 0.5602520492255377
Epoch: 11, Cost: 0.4939438598398527
Epoch: 12, Cost: 0.4441192508152308
Epoch: 13, Cost: 0.40121927514420325
Epoch: 14, Cost: 0.3640109882077388
Epoch: 15, Cost: 0.33167377055339753
Epoch: 16, Cost: 0.3063991898270695
Epoch: 17, Cost: 0.28395800691032125
Epoch: 18, Cost: 0.26428072724414997
Epoch: 19, Cost: 0.24709375047465904
0: ──Rot(-0.92,-0.46,0.46)─╭●──Rot(0.44,0.59,-1.00)──╭●─┤  State
1: ──Rot(1.13,-1.13,0.40)──╰X──Rot(-0.28,0.51,-0.46)─╰X─┤  State

Training 3 qubit circuit
Epoch: 0, Cost: 0.9563788245277092
Epoch: 1, Cost: 0.8745412278816806
Epoch: 2, Cost: 0.7967150136077532

## Training Model (Classical Classifiers)

### Logistic Regression

In [67]:
params = np.load("params/trained_params_2.npy")
#expectation values are treated like classical features 
Xq_train = get_quantum_features(X_train, params)     
Xq_test = get_quantum_features(X_test, params)

q_lg_clf = LogisticRegression().fit(Xq_train, y_train)
q_lg_y_pred = q_lg_clf.predict(Xq_test)  

print("Quantum-feature accuracy:", q_lg_clf.score(Xq_test, y_test))
print("\n\nConfusion Matrix: \n", confusion_matrix(y_test, q_lg_y_pred))    
print("\nClassification Report:\n", classification_report(y_test, q_lg_y_pred))

Quantum-feature accuracy: 0.9238770685579196


Confusion Matrix: 
 [[ 865  115]
 [  46 1089]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.91       980
           1       0.90      0.96      0.93      1135

    accuracy                           0.92      2115
   macro avg       0.93      0.92      0.92      2115
weighted avg       0.93      0.92      0.92      2115



### SVM

In [68]:
q_svm_clf = SVC(kernel='linear', C=1.0, random_state=42)
q_svm_clf.fit(Xq_train, y_train)   

q_svm_y_pred = q_svm_clf.predict(Xq_test)  

print("Quantum-feature accuracy:", q_svm_clf.score(Xq_test, y_test))
print("\n\nConfusion Matrix: \n", confusion_matrix(y_test, q_svm_y_pred))    
print("\nClassification Report:\n", classification_report(y_test, q_svm_y_pred))

Quantum-feature accuracy: 0.9257683215130024


Confusion Matrix: 
 [[ 863  117]
 [  40 1095]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.88      0.92       980
           1       0.90      0.96      0.93      1135

    accuracy                           0.93      2115
   macro avg       0.93      0.92      0.92      2115
weighted avg       0.93      0.93      0.93      2115



### Save Models

In [69]:
joblib.dump(q_lg_clf, "../results/models/quantum_logreg.pkl")
joblib.dump(q_svm_clf, "../results/models/quantum_svm.pkl")

['../results/models/quantum_svm.pkl']

## Experiments 

### Load IBM account

This section of the code cannot run on IE University wifi, as their network (Cisco Umbrella) blocks all outbound DNS. Since we cannot connect to this on university wifi, we will only run noise as 'False' for error free runs. 

In [4]:
import requests, certifi, socket

print("Using cert file:", certifi.where())

try:
    r = requests.get("https://auth.quantum-computing.ibm.com/api/version", verify=certifi.where())
    print("Response:", r.text)
except Exception as e:
    print("Error:", e)

print(socket.gethostbyname("auth.quantum-computing.ibm.com"))


Using cert file: c:\Users\annap\miniconda3\envs\qfm-env\lib\site-packages\certifi\cacert.pem
Error: HTTPSConnectionPool(host='auth.quantum-computing.ibm.com', port=443): Max retries exceeded with url: /api/version (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000024544D65AB0>: Failed to resolve 'auth.quantum-computing.ibm.com' ([Errno 11001] getaddrinfo failed)"))


gaierror: [Errno 11001] getaddrinfo failed

In [2]:
requests.get("https://auth.quantum-computing.ibm.com/api/version")

ConnectionError: HTTPSConnectionPool(host='auth.quantum-computing.ibm.com', port=443): Max retries exceeded with url: /api/version (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000024544237130>: Failed to resolve 'auth.quantum-computing.ibm.com' ([Errno 11001] getaddrinfo failed)"))

In [ ]:
# qiskit.IBMQ / qiskit.providers.ibmq were removed in Qiskit 1.0, and IBM retired the
# old "ibm_quantum" channel in 2025 in favor of the unified "ibm_quantum_platform".
# QiskitRuntimeService replaces both account management and provider/backend lookup.
load_dotenv()
api_key = os.getenv("IBM_API_KEY")
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=api_key, overwrite=True)  # run only once
service = QiskitRuntimeService()

backend = service.least_busy(min_num_qubits=n_qubits, operational=True, simulator=False)

print("Using IBM backend:", backend.name)


### Sweep Qubits (Circuit Hyperparameters)

Looping over a range of circuit hyperparamters (number of qubits, depth (# of layers in var. ansatz), shot (# of times qcircuit is sampled), noise (whether to stimulate device noise - will run on IBM)). 

- More qubits would mean a richer feature space but more noise/cost. 
- More depth/layers would mean higher expressivity but is more difficult to train
- More shots means lower measurement noise but a longer runtime
- More noise means noise realism tradeoff


In [72]:
params = np.load("params/trained_params.npy")      #trained parameters
 
#trained models
q_lg_clf = joblib.load("../results/models/quantum_logreg.pkl")
q_svm_clf = joblib.load("../results/models/quantum_svm.pkl")   


In [82]:
n_qubit_list = [2,3,4]
depth_list = [1,2]
shot_list= [None, 1024]      #0 means no sampling - analytic
noise_list = [False]    #true = use simulator 

In [83]:
lg_results = []
svm_results = []   

for n_qubits, depth, shots, noise in itertools.product(n_qubit_list, depth_list, shot_list, noise_list):
    params_path = f"params/trained_params_{n_qubits}.npy"       #loads correct params file for qubits
    params = np.load(params_path)
    
    #adjust device configration
    if noise:
        dev =  qml.device("qiskit.remote", wires=n_qubits, backend=backend, shots=shots)   #qiskit.ibmq device was removed; qiskit.remote is the current pennylane-qiskit device for a live IBM backend
    else:
        dev = qml.device("default.qubit", wires=n_qubits, shots=shots)


    @qml.qnode(dev)
    def qnode_measure(x, params):
        zz_feature_map(x, wires=range(n_qubits), reps=depth)
        var_ansatz(params)
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    #extract quantum features
    Xq_train = get_quantum_features(X_train, params)
    Xq_test = get_quantum_features(X_test, params)
    # print(f"DEBUG: {Xq_train.mean()}, {Xq_train.std()}")


    q_lr_start = time.time()
    #log reg 
    q_lg_clf = LogisticRegression()
    q_lg_clf.fit(Xq_train, y_train)
    q_lg_y_pred = q_lg_clf.predict(Xq_test)
    q_lg_acc = np.mean(q_lg_y_pred == y_test)
    q_lr_runtime = (time.time()- q_lr_start, 3)


    #svm model 
    q_svm_start = time.time()
    q_svm_clf = SVC(kernel='linear', C=1.0, random_state=42)
    q_svm_clf.fit(Xq_train, y_train)
    q_svm_y_pred = q_svm_clf.predict(Xq_test)   
    q_svm_acc = np.mean(q_svm_y_pred == y_test)
    q_svm_runtime = (time.time()-q_svm_start, 3)

    lg_results.append({
        "n_qubits": n_qubits, 
        "depth": depth,  
        "shots": shots if shots else "analytic", 
        "noise": noise,
        "accuracy": q_lg_acc,
        "runtime_sec": q_lr_runtime[0]
    })    

    svm_results.append({
        "n_qubits": n_qubits,
        "depth": depth,
        "shots": shots if shots else "analytic",
        "noise": noise,   
        "accuracy":q_svm_acc,    
        "runtime_sec": q_svm_runtime[0]   
    })

    print(f"Run ({n_qubits} qubits, depth={depth}, shots={shots}, noise={noise}) = Log reg acc={q_lg_acc:.3f}, SVM acc={q_svm_acc:.3f}")


Run (2 qubits, depth=1, shots=None, noise=False) = Log reg acc=0.814, SVM acc=0.809
Run (2 qubits, depth=1, shots=1024, noise=False) = Log reg acc=0.799, SVM acc=0.809
Run (2 qubits, depth=2, shots=None, noise=False) = Log reg acc=0.809, SVM acc=0.813
Run (2 qubits, depth=2, shots=1024, noise=False) = Log reg acc=0.815, SVM acc=0.807
Run (3 qubits, depth=1, shots=None, noise=False) = Log reg acc=0.915, SVM acc=0.917
Run (3 qubits, depth=1, shots=1024, noise=False) = Log reg acc=0.915, SVM acc=0.916
Run (3 qubits, depth=2, shots=None, noise=False) = Log reg acc=0.861, SVM acc=0.870
Run (3 qubits, depth=2, shots=1024, noise=False) = Log reg acc=0.858, SVM acc=0.867
Run (4 qubits, depth=1, shots=None, noise=False) = Log reg acc=0.852, SVM acc=0.857
Run (4 qubits, depth=1, shots=1024, noise=False) = Log reg acc=0.853, SVM acc=0.854
Run (4 qubits, depth=2, shots=None, noise=False) = Log reg acc=0.703, SVM acc=0.714
Run (4 qubits, depth=2, shots=1024, noise=False) = Log reg acc=0.698, SVM ac

### Save Results

In [79]:
q_lg_df = pd.DataFrame(lg_results)
q_lg_df.to_csv("../results/metrics/q_lr_sweep_results.csv", index=False)
print("\nSaved to results/metrics/q_lr_sweep_results.csv")


q_svm_df = pd.DataFrame(svm_results)
q_svm_df.to_csv("../results/metrics/q_svm_sweep_results.csv", index=False)
print("\nSaved to results/metrics/q_svm_sweep_results.csv")


Saved to results/metrics/q_lr_sweep_results.csv

Saved to results/metrics/q_svm_sweep_results.csv
